# SKKB - Import traces

Load MLflow traces for a single evaluation run directly in Databricks and convert them into the same trace-level flat dataframe shape used by the SKKB data-preparation notebook.

All trace-derived evaluation fields come from the MLflow experiment traces. The helper English columns are enriched separately: `user_query_en` and persisted `expected_response_en` come from the SKKB test-set JSON, and `agent_response_en` comes from a persistent sidecar translation cache.

## Imports

In [0]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [0]:
import config_nbs
config_nbs.add_local_paths()

import importlib
import mlflow
import pandas as pd
import hg_ds_evals.preprocessing.skkb_traces as skkb_traces

importlib.reload(skkb_traces)

build_skkb_dataframe_from_mlflow_search_traces = skkb_traces.build_skkb_dataframe_from_mlflow_search_traces

In [0]:
EXPERIMENT_ID = "471458066277040"
RUN_ID = "3441808bc8db416fbbce3f4878e94d4d" # "361d3ceff6f645d6a5f30bd3bae66fb8" #"3441808bc8db416fbbce3f4878e94d4d"
RUN_NAME = "online_nightly_2026_04_24_infer" #"online_nightly_2026_04_27_infer" # "online_nightly_2026_04_24_infer"
MAX_RESULTS = None

TEST_SET_JSON_PATH = "../input/test_set_513_2026-04-16_14h54_GAI_SK_SK.json"
AGENT_RESPONSE_TRANSLATION_CACHE_PATH = "../input/skkb_mlflow_bot_response_SK_EN.json"  # Legacy cache filename.
KB_SK_CSV_PATH = "../input/KB_GAI_SK_SK_2026-04-20_14h16_phase_1_2.csv"
KB_EN_CSV_PATH = "../input/KB_GAI_SK_EN_2026-04-20_14h16_phase_1_2.csv"

DBX_CATALOG:str = "prod_aut_chat00_lab_catalog"
DBX_SCHEMA:str = "private_uc0115_aix_db"
DBX_TABLE:str = f"dts_eval_skkb_exp_001_{RUN_NAME.replace('-', '_')}"

In [0]:
# Retrieve all traces associated with this evaluation run
traces_df = mlflow.search_traces(
    experiment_ids=[EXPERIMENT_ID],
    run_id=RUN_ID,
    max_results=MAX_RESULTS,
    order_by=["timestamp_ms DESC"],
)

print(f"Shape: {traces_df.shape}")
print(f"Columns: {traces_df.columns.tolist()}")
traces_df.head()

Shape: (484, 12)
Columns: ['trace_id', 'trace', 'client_request_id', 'state', 'request_time', 'execution_duration', 'request', 'response', 'trace_metadata', 'tags', 'spans', 'assessments']


,trace_id,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-3f31a1fc540484398cfab33b318fa7f9,Trace(trace_id=tr-3f31a1fc540484398cfab33b318f...,tr-3f31a1fc540484398cfab33b318fa7f9,TraceState.OK,1777036340022,13618,"{'messages': [['human', 'Potrebujem niečo rieš...",{'messages': [{'content': 'Potrebujem niečo ri...,{'mlflow.sourceRun': '3441808bc8db416fbbce3f48...,{'mlflow.artifactLocation': 'dbfs:/databricks/...,"[{'trace_id': 'PzGh/FQEhDmM+rM7MY+n+Q==', 'spa...","[Expectation(name='test_case_id', source=Asses..."
1,tr-a6d06e5dd3708a1030e247ec3a198f5a,Trace(trace_id=tr-a6d06e5dd3708a1030e247ec3a19...,tr-a6d06e5dd3708a1030e247ec3a198f5a,TraceState.OK,1777036321437,9081,"{'messages': [['human', 'Kde nájdem kontakt na...",{'messages': [{'content': 'Kde nájdem kontakt ...,{'mlflow.sourceRun': '3441808bc8db416fbbce3f48...,{'mlflow.artifactLocation': 'dbfs:/databricks/...,"[{'trace_id': 'ptBuXdNwihAw4kfsOhmPWg==', 'spa...","[Expectation(name='weight', source=AssessmentS..."
2,tr-5a5e53ed79384dd04c5cee843b39f399,Trace(trace_id=tr-5a5e53ed79384dd04c5cee843b39...,tr-5a5e53ed79384dd04c5cee843b39f399,TraceState.OK,1777036294017,17015,"{'messages': [['human', 'Plánujem platiť mobil...",{'messages': [{'content': 'Plánujem platiť mob...,{'mlflow.sourceRun': '3441808bc8db416fbbce3f48...,{'mlflow.artifactLocation': 'dbfs:/databricks/...,"[{'trace_id': 'Wl5T7Xk4TdBMXO6EOznzmQ==', 'spa...","[Expectation(name='weight', source=AssessmentS..."
3,tr-8e00bd30c7e0e5291c9296885c73b044,Trace(trace_id=tr-8e00bd30c7e0e5291c9296885c73...,tr-8e00bd30c7e0e5291c9296885c73b044,TraceState.OK,1777036271472,12743,"{'messages': [['human', 'Ako si tokenizujem ka...",{'messages': [{'content': 'Ako si tokenizujem ...,{'mlflow.sourceRun': '3441808bc8db416fbbce3f48...,{'mlflow.artifactLocation': 'dbfs:/databricks/...,"[{'trace_id': 'jgC9MMfg5SkckpaIXHOwRA==', 'spa...","[Expectation(name='target_enums_to_relevance',..."
4,tr-10d930dfbfed14dbd122f752eef2a271,Trace(trace_id=tr-10d930dfbfed14dbd122f752eef2...,tr-10d930dfbfed14dbd122f752eef2a271,TraceState.OK,1777036244427,17125,"{'messages': [['human', 'Čo je tokenizácia pla...",{'messages': [{'content': 'Čo je tokenizácia p...,{'mlflow.sourceRun': '3441808bc8db416fbbce3f48...,{'mlflow.artifactLocation': 'dbfs:/databricks/...,"[{'trace_id': 'ENkw37/tFNvRIvdS7vKicQ==', 'spa...","[Expectation(name='expected_response', source=..."


In [0]:
parse_result = build_skkb_dataframe_from_mlflow_search_traces(traces_df)

trace_level_df = parse_result.dataframe
parse_errors = parse_result.parse_errors
unmapped_trace_ids = parse_result.unmapped_trace_ids

In [0]:
print(f"Run: {RUN_NAME}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Trace rows fetched: {len(traces_df):,}")
print(f"Parsed rows: {len(trace_level_df):,}")
print(f"Parse errors: {len(parse_errors):,}")
print(f"Rows using trace_id as test_case_id fallback: {len(unmapped_trace_ids):,}")

if parse_errors:
    print(parse_errors[:5])

Run: online_nightly_2026_04_24_infer
Tracking URI: databricks
Trace rows fetched: 484
Parsed rows: 484
Parse errors: 0
Rows using trace_id as test_case_id fallback: 0


In [0]:
trace_level_df.shape

(484, 51)

In [0]:
trace_level_df.columns

Index(['test_case_id', 'trace_id', 'request_time', 'execution_duration_ms',
       'user_query', 'knowledge_search_run_count',
       'knowledge_search_final_run_index', 'knowledge_search_runs',
       'reranked_kb_context', 'kb_version', 'reranked_enum_ids',
       'raw_vector_db_retrieved_candidates_text',
       'raw_vector_db_retrieved_enum_ids',
       'raw_vector_db_retrieved_enum_count', 'raw_vector_db_query_count',
       'raw_vector_db_retrieved_count_by_query', 'raw_vector_db_query_limits',
       'pre_prune_candidates_text', 'pre_prune_enum_ids',
       'pre_prune_enum_count', 'post_prune_candidates_text',
       'post_prune_enum_ids', 'post_prune_enum_count',
       'prune_counts_available', 'prune_candidates_in', 'prune_candidates_out',
       'prune_candidates_dropped', 'agent_response', 'expected_response',
       'expected_enums', 'expected_enums_weights', 'expert_score',
       'expert_rationale', 'enum_relevance_score', 'agents_called',
       'tools_called', 'query_s

In [0]:
trace_level_df.head()

,test_case_id,trace_id,request_time,execution_duration_ms,user_query,knowledge_search_run_count,knowledge_search_final_run_index,knowledge_search_runs,reranked_kb_context,kb_version,reranked_enum_ids,raw_vector_db_retrieved_candidates_text,raw_vector_db_retrieved_enum_ids,raw_vector_db_retrieved_enum_count,raw_vector_db_query_count,raw_vector_db_retrieved_count_by_query,raw_vector_db_query_limits,pre_prune_candidates_text,pre_prune_enum_ids,pre_prune_enum_count,post_prune_candidates_text,post_prune_enum_ids,post_prune_enum_count,prune_counts_available,prune_candidates_in,prune_candidates_out,prune_candidates_dropped,agent_response,expected_response,expected_enums,expected_enums_weights,expert_score,expert_rationale,enum_relevance_score,agents_called,tools_called,query_scope,trace_invariant_violations,reranker_selected_empty,reranker_raw_selected_ids,reranker_valid_selected_ids,reranker_invalid_selected_ids,reranker_unselected_context_ids,reranker_selection_status,reranker_selection_violations,reranker_system_prompt,reranker_user_prompt,reranker_model,reranker_token_usage,main_agent_prompt_hash,daily_banking_agent_prompt_hash
0,Test case 512,tr-3f31a1fc540484398cfab33b318fa7f9,1777036340022,13618,"Potrebujem niečo riešiť ohľadom účtu, môžem na...",1,1,"[{'run_index': 1, 'knowledge_search_span_id': ...",[KB_GAI_SK_SK_2026-04-20_14h16_phase_1_2]\nWRI...,KB_GAI_SK_SK_2026-04-20_14h16_phase_1_2,"[WRITE_MESSAGE, CALL_BRANCH_AUTHORISED, CALL_P...","WRITE_MESSAGE: **Informácie o tom, ako získať ...","[WRITE_MESSAGE, CALL_BRANCH_AUTHORISED, FRAUD@...",180,6,"{'Potrebujem niečo riešiť ohľadom účtu, môžem ...","[30, 30, 30, 30, 30, 30]","WRITE_MESSAGE: **Informácie o tom, ako získať ...","[WRITE_MESSAGE, CALL_BRANCH_AUTHORISED, FRAUD@...",62,SAVING@ACCOUNT_MOVEMENTS: **Vklady na sporiaci...,"[SAVING@ACCOUNT_MOVEMENTS, LOAN@MORTGAGE_APPLI...",18,True,62,18,44,"Áno, môžete. V Georgovi sa dá písať priamo svo...","Ospravedlňujem sa, nemám k dispozícii informác...",[WRITE_MESSAGE],"{""WRITE_MESSAGE"": 4}",6.0,"Expert hovorí, že nevie, či klient môže písať ...",0.8,"[main_agent, daily_banking_agent, llm, tools, ...","[{'name': 'knowledge_search', 'inputs': {'ques...",kb,[],False,"[""WRITE_MESSAGE"", ""CALL_BRANCH_AUTHORISED"", ""C...","[""WRITE_MESSAGE"", ""CALL_BRANCH_AUTHORISED"", ""C...",[],[],ok,[],You are the final selector for a banking knowl...,User question: Potrebujem niečo riešiť ohľadom...,gpt-5-1,"{""input_tokens"": 13695, ""output_tokens"": 44, ""...",8e22b0acb3,8e22b0acb3
1,Test case 511,tr-a6d06e5dd3708a1030e247ec3a198f5a,1777036321437,9081,Kde nájdem kontakt na svojho konzultanta?,1,1,"[{'run_index': 1, 'knowledge_search_span_id': ...",[KB_GAI_SK_SK_2026-04-20_14h16_phase_1_2]\nWRI...,KB_GAI_SK_SK_2026-04-20_14h16_phase_1_2,"[WRITE_MESSAGE, CALL_BRANCH_AUTHORISED, LOAN@M...","WRITE_MESSAGE: **Informácie o tom, ako získať ...","[WRITE_MESSAGE, CALL_BRANCH_AUTHORISED, LOAN@M...",180,6,{'Kde nájdem kontakt na svojho konzultanta?': ...,"[30, 30, 30, 30, 30, 30]","WRITE_MESSAGE: **Informácie o tom, ako získať ...","[WRITE_MESSAGE, CALL_BRANCH_AUTHORISED, LOAN@M...",68,LOAN@CONSOLIDATION_INTEREST_RATE: **Informácie...,"[LOAN@CONSOLIDATION_INTEREST_RATE, EASY_ACCESS...",18,True,68,18,50,Kontakt na svojho konzultanta nájdete priamo v...,"Ospravedlňujem sa, nenašiel som informácie k v...",[WRITE_MESSAGE],"{""WRITE_MESSAGE"": 4}",6.0,The Expert explicitly states they cannot find ...,0.8,"[main_agent, daily_banking_agent, llm, tools, ...","[{'name': 'knowledge_search', 'inputs': {'ques...",kb,[],False,"[""WRITE_MESSAGE"", ""CALL_BRANCH_AUTHORISED"", ""L...","[""WRITE_MESSAGE"", ""CALL_BRANCH_AUTHORISED"", ""L...",[],[],ok,[],You are the final selector for a banking knowl...,User question: Kde nájdem kontakt na svojho ko...,gpt-5-1,"{""input_tokens"": 15510, ""output_tokens"": 45, ""...",8e22b0acb3,8e22b0acb3
2,Test case 510,tr-5a5e53ed79384dd04c5cee843b39f399,1777036294017,17015,"Plánujem platiť mobilom a chcem vedieť, či sú ...",1,1,"[{'

## Add enrichment columns and persistent English text caches

This step restores the full data-preparation table contract needed by the baseline and results notebooks. `user_query_en`, `expected_response_en`, and `categories_list` come from the SKKB test-set JSON; KB attachment columns are looked up from the persisted SK and EN KB CSV snapshots; `agent_response_en` is stored in a sidecar cache keyed by the Slovak agent response text so reruns only translate unseen agent outputs.

In [0]:
import concurrent.futures
import json
from pathlib import Path
from urllib import error, request

TRANSLATE_MODEL = "gpt-5-1"
TRANSLATE_MAX_CONCURRENCY = 20
TRANSLATE_MAX_OUTPUT_TOKENS = 1024

TRANSLATE_SYSTEM_PROMPT = (
    "You are a professional Slovak-to-English translator. Translate the user "
    "message literally and concisely. Preserve banking terminology, product "
    "names, numbers, and formatting. Do NOT add commentary, quotes, or "
    "explanations. Output ONLY the English translation."
 )

_dbx_host = spark.conf.get("spark.databricks.workspaceUrl")
_dbx_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
_translate_url = f"https://{_dbx_host}/serving-endpoints/chat/completions"

def _atomic_write_json(path: str, payload: dict) -> None:
    target_path = Path(path)
    target_path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = target_path.with_suffix(target_path.suffix + ".tmp")
    with temp_path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2, sort_keys=True)
    temp_path.replace(target_path)

def _load_json(path: str, default: dict) -> dict:
    try:
        with open(path, "r", encoding="utf-8") as handle:
            payload = json.load(handle)
    except FileNotFoundError:
        return default
    return payload if isinstance(payload, dict) else default

def _translate_one(text: str) -> str:
    payload = {
        "model": TRANSLATE_MODEL,
        "messages": [
            {"role": "system", "content": TRANSLATE_SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        "max_tokens": TRANSLATE_MAX_OUTPUT_TOKENS,
    }
    req = request.Request(
        _translate_url,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {_dbx_token}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    try:
        with request.urlopen(req, timeout=120) as response:
            response_payload = json.loads(response.read().decode("utf-8"))
    except error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="ignore")
        raise RuntimeError(f"Translation request failed for text {text[:80]!r}: {detail}") from exc

    choice = ((response_payload.get("choices") or [{}])[0].get("message") or {})
    return (choice.get("content") or "").strip()

def _translate_unique_texts(texts: list[str]) -> dict[str, str]:
    unique_texts = sorted({text for text in texts if isinstance(text, str) and text.strip()})
    if not unique_texts:
        return {}
    max_workers = min(TRANSLATE_MAX_CONCURRENCY, len(unique_texts))
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        translations = list(executor.map(_translate_one, unique_texts))
    return dict(zip(unique_texts, translations))

def _attach_kb(ids, lang_table: dict[str, dict[str, str]]) -> list[dict[str, object]]:
    attached = []
    for enum_id in ids if isinstance(ids, list) else []:
        enum_key = str(enum_id)
        kb_row = lang_table.get(enum_key)
        attached.append({
            "enum_id": enum_key,
            "description": (kb_row or {}).get("kb.description", ""),
            "missing": kb_row is None,
        })
    return attached

expected_response_values = trace_level_df["expected_response"].tolist() if "expected_response" in trace_level_df.columns else [""] * len(trace_level_df)
agent_response_values = trace_level_df["agent_response"].tolist() if "agent_response" in trace_level_df.columns else [""] * len(trace_level_df)
reranked_enum_values = trace_level_df["reranked_enum_ids"].tolist() if "reranked_enum_ids" in trace_level_df.columns else [[] for _ in range(len(trace_level_df))]
post_prune_enum_values = trace_level_df["post_prune_enum_ids"].tolist() if "post_prune_enum_ids" in trace_level_df.columns else [[] for _ in range(len(trace_level_df))]

for required_path in (TEST_SET_JSON_PATH, KB_SK_CSV_PATH, KB_EN_CSV_PATH):
    if not Path(required_path).exists():
        raise FileNotFoundError(f"required enrichment file not found: {required_path}")

test_set_payload = _load_json(TEST_SET_JSON_PATH, {})
user_query_to_test_case_id = {}
for test_case_id, record in test_set_payload.items():
    if not isinstance(record, dict):
        continue
    user_query = record.get("user_query")
    if isinstance(user_query, str) and user_query.strip() and user_query not in user_query_to_test_case_id:
        user_query_to_test_case_id[user_query] = test_case_id

def _resolve_test_case_id(row) -> str | None:
    test_case_id = row.get("test_case_id")
    if isinstance(test_case_id, str) and test_case_id in test_set_payload:
        return test_case_id
    user_query = row.get("user_query")
    if isinstance(user_query, str):
        return user_query_to_test_case_id.get(user_query)
    return None

resolved_test_case_ids = trace_level_df.apply(_resolve_test_case_id, axis=1)
matched_test_set_rows = int(resolved_test_case_ids.notna().sum())
print(f"test-set matches: {matched_test_set_rows}/{len(trace_level_df)}")

trace_level_df["user_query_en"] = resolved_test_case_ids.apply(
    lambda test_case_id: ((test_set_payload.get(test_case_id) or {}).get("user_query_en", "")) if isinstance(test_case_id, str) else ""
 )
trace_level_df["categories_list"] = resolved_test_case_ids.apply(
    lambda test_case_id: ((test_set_payload.get(test_case_id) or {}).get("categories_list") or []) if isinstance(test_case_id, str) else []
 )

missing_expected_rows = []
for test_case_id, expected_response in zip(resolved_test_case_ids.tolist(), expected_response_values):
    if not isinstance(test_case_id, str) or not isinstance(expected_response, str) or not expected_response.strip():
        continue
    test_set_record = test_set_payload.get(test_case_id) or {}
    if not (test_set_record.get("expected_response_en") or ""):
        missing_expected_rows.append((test_case_id, expected_response))

print(f"expected_response_en missing in test-set JSON: {len(missing_expected_rows)}")
if missing_expected_rows:
    unique_expected_responses = sorted({text for _, text in missing_expected_rows})
    expected_translations = _translate_unique_texts(unique_expected_responses)
    for test_case_id, expected_response in missing_expected_rows:
        test_set_payload.setdefault(test_case_id, {})["expected_response_en"] = expected_translations.get(expected_response, "")
    _atomic_write_json(TEST_SET_JSON_PATH, test_set_payload)
    print(f"saved expected_response_en updates to {TEST_SET_JSON_PATH}")

trace_level_df["expected_response_en"] = resolved_test_case_ids.apply(
    lambda test_case_id: ((test_set_payload.get(test_case_id) or {}).get("expected_response_en", "")) if isinstance(test_case_id, str) else ""
 )

kb_sk = pd.read_csv(KB_SK_CSV_PATH, sep="|", dtype=str).fillna("")
kb_en = pd.read_csv(KB_EN_CSV_PATH, sep="|", dtype=str).fillna("")
sk_by_id = kb_sk.set_index("kb.knowledgeId").to_dict(orient="index")
en_by_id = kb_en.set_index("kb.knowledgeId").to_dict(orient="index")
trace_level_df["reranked_enums_kb_sk"] = [_attach_kb(ids, sk_by_id) for ids in reranked_enum_values]
trace_level_df["reranked_enums_kb_en"] = [_attach_kb(ids, en_by_id) for ids in reranked_enum_values]
trace_level_df["post_prune_candidates_kb_sk"] = [_attach_kb(ids, sk_by_id) for ids in post_prune_enum_values]
trace_level_df["post_prune_candidates_kb_en"] = [_attach_kb(ids, en_by_id) for ids in post_prune_enum_values]

agent_translation_cache = _load_json(AGENT_RESPONSE_TRANSLATION_CACHE_PATH, {"agent_response": {}})
agent_response_map = dict(agent_translation_cache.get("agent_response") or {})
unique_agent_responses = sorted({
    text
    for text in agent_response_values
    if isinstance(text, str) and text.strip()
})
missing_agent_responses = [text for text in unique_agent_responses if text not in agent_response_map]
print(f"agent_response_en cached: {len(unique_agent_responses) - len(missing_agent_responses)}/{len(unique_agent_responses)}")
if missing_agent_responses:
    agent_response_map.update(_translate_unique_texts(missing_agent_responses))
    _atomic_write_json(AGENT_RESPONSE_TRANSLATION_CACHE_PATH, {"agent_response": agent_response_map})
    print(f"saved agent_response_en cache to {AGENT_RESPONSE_TRANSLATION_CACHE_PATH}")

trace_level_df["agent_response_en"] = [
    agent_response_map.get(text, "") if isinstance(text, str) and text.strip() else ""
    for text in agent_response_values
 ]

print(f"user_query_en non-empty: {(trace_level_df['user_query_en'].str.len() > 0).sum()}/{len(trace_level_df)}")
print(f"categories_list populated: {sum(bool(value) for value in trace_level_df['categories_list'])}/{len(trace_level_df)}")
print(f"expected_response_en non-empty: {(trace_level_df['expected_response_en'].str.len() > 0).sum()}/{len(trace_level_df)}")
print(f"agent_response_en non-empty: {(trace_level_df['agent_response_en'].str.len() > 0).sum()}/{len(trace_level_df)}")
print(f"reranked_enums_kb_sk populated: {sum(bool(value) for value in trace_level_df['reranked_enums_kb_sk'])}/{len(trace_level_df)}")
print(f"post_prune_candidates_kb_sk populated: {sum(bool(value) for value in trace_level_df['post_prune_candidates_kb_sk'])}/{len(trace_level_df)}")

test-set matches: 484/484
expected_response_en missing in test-set JSON: 0
agent_response_en cached: 484/484
user_query_en non-empty: 403/484
categories_list populated: 484/484
expected_response_en non-empty: 484/484
agent_response_en non-empty: 484/484
reranked_enums_kb_sk populated: 432/484
post_prune_candidates_kb_sk populated: 434/484


In [0]:
print(f"Run: {RUN_NAME}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Trace rows fetched: {len(traces_df):,}")
print(f"Parsed rows: {len(trace_level_df):,}")
print(f"Parse errors: {len(parse_errors):,}")
print(f"Rows using trace_id as test_case_id fallback: {len(unmapped_trace_ids):,}")

if parse_errors:
    print(parse_errors[:5])

Run: online_nightly_2026_04_24_infer
Tracking URI: databricks
Trace rows fetched: 484
Parsed rows: 484
Parse errors: 0
Rows using trace_id as test_case_id fallback: 0


In [0]:
trace_level_df.iloc[0][["user_query", "user_query_en", "agent_response", "agent_response_en", "expected_response", "expected_response_en"]]

user_query              Potrebujem niečo riešiť ohľadom účtu, môžem na...
user_query_en                                                            
agent_response          Áno, môžete. V Georgovi sa dá písať priamo svo...
agent_response_en       Yes, you can. In George, you can write directl...
expected_response       Ospravedlňujem sa, nemám k dispozícii informác...
expected_response_en    I apologize, I do not have information on whet...
Name: 0, dtype: object

## View data

In [0]:
display(trace_level_df)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
cols = ["test_case_id", 
        "user_query", "user_query_en",
        "raw_vector_db_retrieved_enum_count", "pre_prune_enum_count", "post_prune_enum_count", "reranked_enum_ids", "expected_enums",
        'reranker_raw_selected_ids', 'reranker_valid_selected_ids', 'reranker_invalid_selected_ids', 'reranker_unselected_context_ids', 'reranker_selection_status', 'reranker_selection_violations']
missing_cols = [column for column in cols if column not in trace_level_df.columns]
if missing_cols:
    print(f"Skipping missing preview columns: {missing_cols}")
cols = [column for column in cols if column in trace_level_df.columns]
display(trace_level_df[cols].sort_values(by="test_case_id", ascending=True).head(25))

test_case_id,user_query,user_query_en,raw_vector_db_retrieved_enum_count,pre_prune_enum_count,post_prune_enum_count,reranked_enum_ids,expected_enums,reranker_raw_selected_ids,reranker_valid_selected_ids,reranker_invalid_selected_ids,reranker_unselected_context_ids,reranker_selection_status,reranker_selection_violations
Test case 0,"Chcem si poslať peniaze zo sporenia späť na účet, ako na to?",How can I transfer money from my savings back to my account?,180,77,18,List(SAVING@ACCOUNT_MOVEMENTS),List(SAVING@ACCOUNT_MOVEMENTS),"[""SAVING@ACCOUNT_MOVEMENTS""]","[""SAVING@ACCOUNT_MOVEMENTS""]",[],[],ok,[]
Test case 1,"Dá sa na sporenie vložiť aj hotovosť, alebo len prevodom?","Can I deposit cash into my savings, or only via transfer?",180,68,18,"List(SAVING@ACCOUNT_MOVEMENTS, ATM@DEPOSIT)",List(SAVING@ACCOUNT_MOVEMENTS),"[""SAVING@ACCOUNT_MOVEMENTS"", ""ATM@DEPOSIT""]","[""SAVING@ACCOUNT_MOVEMENTS"", ""ATM@DEPOSIT""]",[],[],ok,[]
Test case 10,Koľko sporení si môžem naraz vytvoriť a v akej mene to je?,"How many savings accounts can I create at once, and in what currency are they?",180,70,18,List(SAVING@ABOUT),List(SAVING@ABOUT),"[""SAVING@ABOUT""]","[""SAVING@ABOUT""]",[],[],ok,[]
Test case 100,"Kde si viem pozrieť, či sa mi odmena vôbec zaevidovala v histórii?",Where can I check whether my reward has been recorded in the history at all?,180,75,18,List(DATEIO@CLAIMS),List(DATEIO@CLAIMS),"[""DATEIO@CLAIMS""]","[""DATEIO@CLAIMS""]",[],[],ok,[]
Test case 101,Ako reklamujem Moneyback?,How do I file a claim for Moneyback?,180,74,18,List(DATEIO@CLAIMS),List(DATEIO@CLAIMS),"[""DATEIO@CLAIMS""]","[""DATEIO@CLAIMS""]",[],[],ok,[]
Test case 102,"Chcem niekomu dať prístup k účtu, čo všetko bude môcť robiť?",I want to give someone access to my account—what will they be able to do?,180,78,18,List(ACCOUNT@DISPOSING_PERSON),List(ACCOUNT@DISPOSING_PERSON),"[""ACCOUNT@DISPOSING_PERSON""]","[""ACCOUNT@DISPOSING_PERSON""]",[],[],ok,[]
Test case 103,"Keď majiteľ účtu zomrie, platí disponent ešte ďalej alebo sa to zruší?","If the account owner dies, does the authorized person remain valid or is it cancelled?",180,53,18,List(ACCOUNT@DISPOSING_PERSON),List(ACCOUNT@DISPOSING_PERSON),"[""ACCOUNT@DISPOSING_PERSON""]","[""ACCOUNT@DISPOSING_PERSON""]",[],[],ok,[]
Test case 104,Kto je disponent?,What is an authorized person (disponent)?,0,0,0,List(),List(ACCOUNT@DISPOSING_PERSON),[],[],[],[],not_applicable,[]
Test case 105,Pre koho je SPACE účet a čo je na ňom najväčšia výhoda?,Who is the SPACE account for and what is its biggest advantage?,180,53,18,List(GIRO@STANDARD_ABOUT),List(GIRO@STANDARD_ABOUT),"[""GIRO@STANDARD_ABOUT""]","[""GIRO@STANDARD_ABOUT""]",[],[],ok,[]
Test case 106,"Dá sa spraviť spoločný účet pre dvoch majiteľov, alebo len disponenti?","Can I open a joint account for two owners, or only add authorized persons?",180,60,18,"List(GIRO@STANDARD_ABOUT, GIRO@FOREIGN_DISPOSING_PERSON, GIRO@FOREIGN_ABOUT)",List(GIRO@STANDARD_ABOUT),"[""GIRO@STANDARD_ABOUT"", ""GIRO@FOREIGN_DISPOSING_PERSON"", ""GIRO@FOREIGN_ABOUT""]","[""GIRO@STANDARD_ABOUT"", ""GIRO@FOREIGN_DISPOSING_PERSON"", ""GIRO@FOREIGN_ABOUT""]",[],[],ok,[]


In [0]:
for enum_column in ("post_prune_enum_ids", "pre_prune_enum_ids"):
    if enum_column in trace_level_df.columns:
        print(f"empty {enum_column}:", sum(trace_level_df[enum_column].apply(lambda x: x == [])))
    else:
        print(f"{enum_column} missing")
count_cols = [
    "knowledge_search_run_count",
    "raw_vector_db_query_count",
    "raw_vector_db_retrieved_enum_count",
    "pre_prune_enum_count",
    "post_prune_enum_count",
    "prune_candidates_in",
    "prune_candidates_out",
    "prune_candidates_dropped",
]
count_cols = [column for column in count_cols if column in trace_level_df.columns]
if count_cols:
    display(trace_level_df[["test_case_id", *count_cols]].describe())

empty post_prune_enum_ids: 50
empty pre_prune_enum_ids: 50


knowledge_search_run_count,raw_vector_db_query_count,raw_vector_db_retrieved_enum_count,pre_prune_enum_count,post_prune_enum_count,prune_candidates_in,prune_candidates_out,prune_candidates_dropped
484.0,484.0,484.0,484.0,484.0,484.0,484.0,484.0
0.8966942148760331,5.1115702479338845,153.34710743801654,55.21487603305785,16.363636363636363,55.21487603305785,16.363636363636363,38.85123966942149
0.3046727557271984,2.128372470190921,63.85117410572765,23.281233740892212,5.5913272090436745,23.281233740892212,5.5913272090436745,19.143204971913242
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1.0,6.0,180.0,47.75,18.0,47.75,18.0,29.0
1.0,6.0,180.0,60.0,18.0,60.0,18.0,42.0
1.0,6.0,180.0,70.0,18.0,70.0,18.0,52.0
1.0,7.0,210.0,97.0,22.0,97.0,22.0,79.0


### Missing EN

In [0]:
print(sum(pd.isna(trace_level_df["user_query_en"])))
print(sum(trace_level_df["user_query_en"] == ""))

0
81


### Execution time stats

In [0]:
trace_level_df["execution_duration_s"] = trace_level_df["execution_duration_ms"] / 1000
trace_level_df["execution_duration_min"] = trace_level_df["execution_duration_ms"] / (1000 * 60)

trace_level_df["execution_duration_s"].describe()

count    484.000000
mean      14.115128
std        5.176139
min        4.248000
25%       10.667000
50%       12.830500
75%       16.779000
max       36.816000
Name: execution_duration_s, dtype: float64

### Query Scope distribution

In [0]:
if "query_scope" in trace_level_df.columns:
    display(trace_level_df["query_scope"].value_counts(dropna=False))
else:
    print("query_scope missing")

query_scope
kb                 406
main_agent          55
banking_data        13
rule1_violation     10
Name: count, dtype: int64

### Extract traces with over 1 min duration

In [0]:
cond = trace_level_df["execution_duration_s"] > 60
traces_1min = trace_level_df[cond].copy()
traces_1min.shape
traces_1min.to_csv("traces_1min.csv", index=False)

## Save parsed traces to Unity catalog

In [0]:
if trace_level_df.empty:
    raise ValueError("trace_level_df is empty; nothing to write")

trace_level_to_write = trace_level_df.copy()

json_string_columns = [
    "reranked_enum_ids",
    "expected_enums",
    "knowledge_search_runs",
    "raw_vector_db_retrieved_enum_ids",
    "raw_vector_db_retrieved_count_by_query",
    "raw_vector_db_query_limits",
    "pre_prune_enum_ids",
    "post_prune_enum_ids",
    "categories_list",
    "reranked_enums_kb_sk",
    "reranked_enums_kb_en",
    "post_prune_candidates_kb_sk",
    "post_prune_candidates_kb_en",
    "agents_called",
    "tools_called",
    "trace_invariant_violations",
]

for column in json_string_columns:
    if column in trace_level_to_write.columns:
        trace_level_to_write[column] = trace_level_to_write[column].apply(
            lambda value: json.dumps(value, ensure_ascii=False)
            if isinstance(value, (list, dict))
            else (value if isinstance(value, str) else "[]")
        )

sdf = spark.createDataFrame(trace_level_to_write)
row_count = len(trace_level_to_write)

full_table = f"{DBX_CATALOG}.{DBX_SCHEMA}.{DBX_TABLE}"
(
    sdf
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table)
)

print(f"wrote {row_count} rows to {full_table}")

wrote 484 rows to prod_aut_chat00_lab_catalog.private_uc0115_aix_db.dts_eval_skkb_exp_001_online_nightly_2026_04_24_infer


## Verify table

In [0]:
rb = spark.read.table(f"{DBX_CATALOG}.{DBX_SCHEMA}.{DBX_TABLE}")
print(f"read back row count: {rb.count()}")
rb.printSchema()
preview_cols = [
    "test_case_id",
    "query_scope",
    "user_query",
    "user_query_en",
    "expected_response_en",
    "agent_response_en",
    "expert_score",
    "enum_relevance_score",
    "reranker_model",
    "reranker_raw_selected_ids",
    "reranker_selection_status",
    "reranker_invalid_selected_ids",
    "reranker_unselected_context_ids",
    "reranker_selected_empty",
    "post_prune_enum_ids",
    "agents_called",
    "tools_called",
]
missing_preview_cols = [column for column in preview_cols if column not in rb.columns]
if missing_preview_cols:
    print(f"Skipping missing read-back columns: {missing_preview_cols}")
rb.select(*[column for column in preview_cols if column in rb.columns]).show(5, truncate=120)

read back row count: 484
root
 |-- test_case_id: string (nullable = true)
 |-- trace_id: string (nullable = true)
 |-- request_time: long (nullable = true)
 |-- execution_duration_ms: long (nullable = true)
 |-- user_query: string (nullable = true)
 |-- knowledge_search_run_count: long (nullable = true)
 |-- knowledge_search_final_run_index: long (nullable = true)
 |-- knowledge_search_runs: string (nullable = true)
 |-- reranked_kb_context: string (nullable = true)
 |-- kb_version: string (nullable = true)
 |-- reranked_enum_ids: string (nullable = true)
 |-- raw_vector_db_retrieved_candidates_text: string (nullable = true)
 |-- raw_vector_db_retrieved_enum_ids: string (nullable = true)
 |-- raw_vector_db_retrieved_enum_count: long (nullable = true)
 |-- raw_vector_db_query_count: long (nullable = true)
 |-- raw_vector_db_retrieved_count_by_query: string (nullable = true)
 |-- raw_vector_db_query_limits: string (nullable = true)
 |-- pre_prune_candidates_text: string (nullable = true)